# Live-data RAG: a vector store you can write to

**Infino is a vector store you can write to.** Append new items, upsert by id,
delete — and both retrieval *and* SQL reflect every change immediately, with no
reindex, and survive a reopen from object storage.

This example keeps a product catalog live: index it, answer a question, add new
stock, mark an item down, clear some out, then reopen from disk and confirm
every change persisted. The retrieval path runs **key-free**; set an LLM key
(see the README) to also generate answers — otherwise the notebook prints the
retrieved context.

In [1]:
import sys
import shutil
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
import pyarrow as pa
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_infino import InfinoVectorStore

from _shared.lc import MiniLMEmbeddings, chat_model
from _shared.embedding import DIM, METRIC
from _shared.loaders import load_amazon
from _shared.sql import query

DB_DIR = "./live_inventory_data"
TABLE = "products"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

embeddings = MiniLMEmbeddings()
llm = chat_model()  # None when no key is set; the demo still runs (prints context)
print("LLM answers:", "on" if llm else "off (retrieval only)")

LLM answers: on


## Build the store

`InfinoVectorStore.from_texts` creates one Infino table and indexes the catalog.
`price` and `rating` are **promoted to scalar columns** (`metadata_columns`) so
they're filterable in SQL and attached to every retrieved `Document`; `title`
and `category` ride along in the JSON metadata. We pass stable `ids` so we can
upsert and delete specific products later.

In [2]:
products = load_amazon(n=600)
ids = [f"p-{i}" for i in range(len(products))]
texts = [p["text"] for p in products]
metadatas = [
    {"title": p["title"], "category": p["category"],
     "price": p["price"], "rating": p["rating"]}
    for p in products
]
metadata_columns = [
    pa.field("price", pa.float64(), nullable=False),
    pa.field("rating", pa.float64(), nullable=False),
]

db = infino.connect(DB_DIR)
store = InfinoVectorStore.from_texts(
    texts, embeddings, metadatas,
    connection=db, table_name=TABLE, dim=DIM, metric=METRIC,
    ids=ids, metadata_columns=metadata_columns,
)
print("indexed", query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0], "products")

indexed 600 products


## RAG over the live catalog

`store.as_retriever()` returns a standard LangChain `BaseRetriever`, so the chain
below is plain LCEL. We ask for a product the catalog doesn't carry yet — note
the answer.

In [3]:
retriever = store.as_retriever(search_kwargs={"k": 4})

PROMPT = ChatPromptTemplate.from_template(
    "You are a shopping assistant. Answer using only the catalog context.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)


def format_docs(docs) -> str:
    return "\n".join(
        f"- {d.metadata.get('title', '')} "
        f"(${d.metadata.get('price')}, {d.metadata.get('rating')}★)"
        for d in docs
    )


def answer(question: str) -> None:
    docs = retriever.invoke(question)
    print(f"Q: {question}\nretrieved:\n{format_docs(docs)}")
    if llm is not None:
        chain = PROMPT | llm | StrOutputParser()
        print("answer:", chain.invoke(
            {"context": format_docs(docs), "question": question}))


QUESTION = "wireless earbuds for running, sweatproof"
answer(QUESTION)  # baseline — no such product in the catalog yet

Q: wireless earbuds for running, sweatproof
retrieved:
- WURGIIX Ribbed Stretch Bandie Women's Headbands Headwraps Hair Bands Bows Hair Accessories（Wine red）Mother's day gift ($5.99, 4.1★)
- Big Bunny Ears Headband for Easter,Halloween Party Costume Accessories/Sweet Sexy Rabbit Ear Hair(Black) ($6.99, 4.3★)
- Picoway 20 Pack Mouse Ears Solid Black and Red Bow Headband ($15.99, 4.8★)
- Ariel Ear Headband Ariel Mouse Ears Headband Ariel Minnie Ears Headband ($39.99, 4.8★)


answer: I couldn’t find any wireless earbuds in the catalog.

Available items are only headbands/accessories:
- WURGIIX Ribbed Stretch Bandie Women's Headbands Headwraps Hair Bands Bows Hair Accessories（Wine red） — $5.99, 4.1★
- Big Bunny Ears Headband for Easter,Halloween Party Costume Accessories/Sweet Sexy Rabbit Ear Hair(Black) — $6.99, 4.3★
- Picoway 20 Pack Mouse Ears Solid Black and Red Bow Headband — $15.99, 4.8★
- Ariel Ear Headband Ariel Mouse Ears Headband Ariel Minnie Ears Headband — $39.99, 4.8★


## Append new stock — searchable immediately

`add_texts` embeds and commits the new product. Re-ask the *same* question: the
just-added item is now the top hit, live for search the moment it's added.

In [4]:
store.add_texts(
    ["NovaSound Z9 wireless running earbuds, sweatproof, 30h battery"],
    metadatas=[{"title": "NovaSound Z9", "category": "Electronics",
                "price": 79.0, "rating": 4.7}],
    ids=["sku-novasound-z9"],
)
answer(QUESTION)

Q: wireless earbuds for running, sweatproof
retrieved:
- NovaSound Z9 ($79.0, 4.7★)
- WURGIIX Ribbed Stretch Bandie Women's Headbands Headwraps Hair Bands Bows Hair Accessories（Wine red）Mother's day gift ($5.99, 4.1★)
- Big Bunny Ears Headband for Easter,Halloween Party Costume Accessories/Sweet Sexy Rabbit Ear Hair(Black) ($6.99, 4.3★)
- Picoway 20 Pack Mouse Ears Solid Black and Red Bow Headband ($15.99, 4.8★)


answer: NovaSound Z9 — $79.0, 4.7★


## Update = upsert by id

Re-adding the same `id` overwrites the row (Infino superfiles are immutable, so
an upsert is delete-then-append under the hood). Mark the price down: the change
shows up, and the row count is **unchanged** — it's an update, not a duplicate.

In [5]:
total_before = query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
store.add_texts(
    ["NovaSound Z9 wireless running earbuds, sweatproof, 30h battery"],
    metadatas=[{"title": "NovaSound Z9", "category": "Electronics",
                "price": 49.0, "rating": 4.7}],
    ids=["sku-novasound-z9"],  # same id -> overwrite, not a second row
)
total_after = query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
doc = store.get_by_ids(["sku-novasound-z9"])[0]
print(f"price -> ${doc.metadata.get('price')}, "
      f"row count {total_before} -> {total_after} (unchanged)")

price -> $49.0, row count 601 -> 601 (unchanged)


## Delete (clearance)

`delete(ids=...)` removes rows from retrieval **and** SQL at once.

In [6]:
before = query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
store.delete(["p-0", "p-1", "p-2"])
after = query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
print(f"cleared 3 -> rows {before} then {after}")
print("p-0 still present:", bool(store.get_by_ids(["p-0"])))

cleared 3 -> rows 601 then 598
p-0 still present: False


## Durability — reopen from disk

Everything committed lives on disk (object storage in production). Drop the
handles, reconnect, and reopen the table with the same `InfinoVectorStore`
constructor: the markdown, the new stock, and the deletions are all still there.
The full-text index and SQL views stay correct after every mutation.

In [7]:
del store, db  # drop the handles, simulate a fresh process

db2 = infino.connect(DB_DIR)
store2 = InfinoVectorStore(
    db2, TABLE, embeddings, dim=DIM, metric=METRIC,
    metadata_columns=metadata_columns,
)
doc = store2.get_by_ids(["sku-novasound-z9"])[0]
total = query(db2, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
print(f"reopened: {total} products; NovaSound Z9 still ${doc.metadata.get('price')}")

reopened: 598 products; NovaSound Z9 still $49.0


## Takeaway

One Infino table served as a **mutable, durable** LangChain vector store: append,
upsert, and delete flowed straight through to vector retrieval and SQL, and
survived a reopen from disk. The same table also answers BM25, hybrid, and SQL
queries (see the other examples).